# Bayesian fitting of QENS data: from the posterior to derived quantities

A least-squares fit reports one best value and one symmetric uncertainty for each parameter. A **Bayesian**
analysis reports the whole **posterior distribution**: how probable each combination of parameter values is,
given the data. This notebook samples the posterior of a QENS model with the DREAM algorithm, and then
compares two ways of getting the uncertainty of a quantity that is derived from the parameters: the first-order
propagation of `es.math`, and the posterior itself.

The model and the data are those of the notebook on fitting QENS data with a Lorentzian, which is the better
place to start. Each part here closes with a few exercises on what was just shown. Their hints and solutions
are folded away and open on a click, so the text reads straight through; a solution is a worked answer with its
result.

In [1]:
import numpy as np
import plotly.graph_objects as go
import pooch
from scipy.stats import norm

import easyscience as es
from easyscience import AvailableMinimizers
from easyscience import DescriptorNumber
from easyscience import Fitter
from easyscience import Parameter
from easyscience.base_classes import ModelBase
from easyscience.fitting import Sampler

The function `show` prints a labelled number as value, uncertainty and unit.

In [2]:
def show(label, number):
    """Print a labelled number as value ± uncertainty with its unit."""
    print(f'{label} = {number.value:.4g} ± {number.error or 0.0:.2g} {number.unit}')

## The data and the model

The data are simulated and reduced QENS intensities, as in the notebook on fitting QENS data, for the energy transfer $\omega$ in meV,
cropped to $|\omega| < 0.06$ meV:

In [3]:
filename = pooch.retrieve(
    url='https://public.esss.dk/groups/scipp/dmsc-summer-school/2025/4-reduction/energy_transfer_QENS_unknown_quasi_elastic_many_neutrons.dat',
    known_hash='sha256:e49fa9a1d2ef5eeb714524903e8ffa9de6616e5a299a799cbb0b2f3ee8fd459a',
)
omega, intensity, error = np.loadtxt(filename, unpack=True)
keep = np.isfinite(intensity) & (omega > -0.06) & (omega < 0.06)
omega, intensity, error = omega[keep], intensity[keep], error[keep]

print(f'{omega.size} points')

120 points


The model is a Lorentzian with area $A$, half width $\gamma$ and centre $\omega_0$, convolved with a Gaussian
resolution of width $\sigma$. The `min` and `max` of every parameter are not only the range of a fit: for the
sampler they define a **uniform prior**, so the sampler only visits values inside the range.

In [4]:
class Quasielastic(ModelBase):
    """A Lorentzian convolved with a Gaussian resolution, as a function of the energy transfer."""

    def __init__(self, area: float, gamma: float, omega_0: float, sigma: float):
        super().__init__()
        self._area = Parameter(name='area', value=area, min=0.01, max=100)
        self._gamma = Parameter(name='gamma', value=gamma, unit='meV', min=1e-4, max=1e-2)
        self._omega_0 = Parameter(name='omega_0', value=omega_0, unit='meV', min=0, max=2e-3)
        self._sigma = Parameter(name='sigma', value=sigma, unit='meV', min=1e-5, max=1e-1)

    @property
    def area(self) -> Parameter:
        return self._area

    @property
    def gamma(self) -> Parameter:
        return self._gamma

    @property
    def omega_0(self) -> Parameter:
        return self._omega_0

    @property
    def sigma(self) -> Parameter:
        return self._sigma

    def __call__(self, omega):
        width = self._gamma.value
        lorentzian = (
            self._area.value / np.pi * width / ((omega - self._omega_0.value) ** 2 + width**2)
        )
        resolution = norm(0, self._sigma.value).pdf(omega)
        resolution /= resolution.sum()
        return np.convolve(lorentzian, resolution, 'same')

## Start from a fit

The sampler starts its chains close to the current parameter values, so it is worth fitting first. The fit
also gives the reference for the comparison below: a value and a first-order uncertainty for every parameter.
The sampler moves the parameters while it works, so the fitted numbers are kept in separate
`DescriptorNumber`s:

In [5]:
model = Quasielastic(area=10.0, gamma=8e-3, omega_0=1e-3, sigma=1e-3)
fitter = Fitter(model, model)
fitter.fit(x=omega, y=intensity, weights=1 / error)

fitted = {
    parameter.name: DescriptorNumber(
        name=parameter.name,
        value=parameter.value,
        unit=parameter.unit,
        variance=parameter.variance,
    )
    for parameter in (model.area, model.gamma, model.omega_0, model.sigma)
}
for name, number in fitted.items():
    show(name, number)

area = 5.227 ± 0.05 dimensionless
gamma = 0.007161 ± 0.00014 meV
omega_0 = 0.001142 ± 9.4e-05 meV
sigma = 0.001087 ± 0.00059 meV


## Sampling the posterior

DREAM is an ensemble Markov-chain Monte Carlo method: many chains explore the parameter space in parallel
and exchange information. The sampler takes the configured fitter, the data and
the weights, and returns the draws. The sampler requires a Bumps minimizer, so the fitter is switched to it
first. The run takes a few tens of seconds.

In [6]:
fitter.switch_minimizer(AvailableMinimizers.Bumps)
sampler = Sampler(fitter, omega, intensity, weights=1 / error)
results = sampler.sample(samples=20000, burn=500, thin=2)

columns = {
    parameter.name: results.draws[:, results.param_names.index(parameter.unique_name)]
    for parameter in (model.area, model.gamma, model.omega_0, model.sigma)
}
print(f'{results.draws.shape[0]} draws for {results.draws.shape[1]} parameters')

8240 draws for 4 parameters


Before the draws are used, the chains have to be checked for convergence. The Gelman-Rubin statistic
$\hat{R}$ compares the variation within each chain with the variation between chains. Values close to 1 mean that the chains agree: below 1.1 is acceptable and below 1.01 is strict. If a value is higher, draw more samples:

In [7]:
for name, value in zip(results.param_names, results.state.gelman()):
    parameter = next(
        p for p in (model.area, model.gamma, model.omega_0, model.sigma) if p.unique_name == name
    )
    print(f'R-hat of {parameter.name:8s} {value:.3f}')

R-hat of area     1.016
R-hat of gamma    1.023
R-hat of omega_0  1.019
R-hat of sigma    1.017


The median and the interval that holds the central 68 % of the draws summarise each marginal posterior. They
can be compared with the fitted value and its symmetric uncertainty:

In [8]:
for name, draws in columns.items():
    low, median, high = np.percentile(draws, [16, 50, 84])
    number = fitted[name]
    print(
        f'{name:8s} fit {number.value:.4g} ± {number.error:.2g}   posterior {median:.4g} (+{high - median:.2g} / -{median - low:.2g})'
    )

area     fit 5.227 ± 0.05   posterior 5.224 (+0.058 / -0.053)
gamma    fit 0.007161 ± 0.00014   posterior 0.007148 (+0.00014 / -0.00017)
omega_0  fit 0.001142 ± 9.4e-05   posterior 0.001134 (+0.00011 / -0.00011)
sigma    fit 0.001087 ± 0.00059   posterior 0.001116 (+0.00061 / -0.0005)


The draws also show how the parameters vary together. The correlation coefficient of two parameters lies
between $-1$ and $+1$; a value far from 0 means that the data cannot determine the two independently. The
first-order arithmetic of `es.math` ignores this.

In [9]:
names = list(columns)
correlation = np.corrcoef([columns[name] for name in names])
for i, first in enumerate(names):
    for j in range(i + 1, len(names)):
        print(f'correlation of {first} and {names[j]}: {correlation[i, j]:+.2f}')

correlation of area and gamma: +0.15
correlation of area and omega_0: +0.03
correlation of area and sigma: -0.10
correlation of gamma and omega_0: -0.10
correlation of gamma and sigma: -0.68
correlation of omega_0 and sigma: +0.20


## A derived quantity from the draws

Every draw is a possible set of parameters, so a derived quantity can be computed for every draw. The
distribution of the results is the posterior of that quantity, with no approximation. The relaxation time is
$\tau = \hbar/\gamma$. On the draws this is plain NumPy, because `es.math` takes one `Parameter` at a time.
The first-order propagation of `es.math` starts from the fitted value:

In [10]:
hbar = DescriptorNumber(name='hbar', value=0.6582119569, unit='meV*ps', variance=0.0)

tau_first_order = hbar / fitted['gamma']
tau_draws = hbar.value / columns['gamma']
low, median, high = np.percentile(tau_draws, [16, 50, 84])

show('relaxation time, first order', tau_first_order)
print(f'relaxation time, posterior = {median:.4g} (+{high - median:.2g} / -{median - low:.2g}) ps')

relaxation time, first order = 91.92 ± 1.8 ps
relaxation time, posterior = 92.09 (+2.2 / -1.8) ps


The width $\gamma$ is determined well, so the two agree closely. The chart shows the posterior of $\tau$ with
the Gaussian that the first-order uncertainty describes:

In [11]:
import base64
import contextlib
import html
import io
import re

from IPython import get_ipython
from IPython.display import HTML
from IPython.display import IFrame
from IPython.display import display
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import PythonLexer


def show_figure(fig: go.Figure, height: int = 420) -> None:
    """Show a Plotly figure as an embedded iframe, so that it renders in the docs."""
    page = (
        go
        .Figure(fig)
        .update_layout(template='plotly_white')
        .to_html(
            full_html=True,
            include_plotlyjs='cdn',
            default_width='100%',
            default_height=f'{height - 20}px',
        )
    )
    encoded = base64.b64encode(page.encode('utf-8')).decode('ascii')
    display(IFrame(src=f'data:text/html;base64,{encoded}', width='100%', height=height))


def solution(note: str, source: str) -> None:
    """Run a solution in a copy of the notebook namespace and show it, folded, with its output."""
    code = source.strip('\n')
    namespace = dict(get_ipython().user_ns)
    stream = io.StringIO()
    with contextlib.redirect_stdout(stream):
        exec(code, namespace)
    parts = [
        '<details class="example"><summary>Solution</summary>',
        highlight(code, PythonLexer(), HtmlFormatter()),
    ]
    result = stream.getvalue().rstrip('\n')
    if result:
        parts.append('<p>Output:</p><pre>' + html.escape(result) + '</pre>')
    if note.strip():
        text = re.sub('`([^`]+)`', r'<code>\1</code>', html.escape(note.strip()))
        parts.append('<p>' + text + '</p>')
    parts.append('</details>')
    display(HTML(''.join(parts)))


get_ipython().register_magic_function(solution, magic_kind='cell')

In [12]:
grid = np.linspace(tau_draws.min(), tau_draws.max(), 300)
gaussian = norm(tau_first_order.value, tau_first_order.error).pdf(grid)

fig = go.Figure()
fig.add_trace(
    go.Histogram(
        x=tau_draws, histnorm='probability density', nbinsx=60, name='posterior', opacity=0.6
    )
)
fig.add_trace(
    go.Scatter(x=grid, y=gaussian, mode='lines', name='first order', line=dict(color='black'))
)
fig.update_layout(
    xaxis_title='relaxation time / ps',
    yaxis_title='probability density',
    bargap=0.02,
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.7, y=0.95),
)
show_figure(fig)

### Exercise 1: The half-life from the posterior

The intermediate scattering function falls to one half at $t_{1/2} = \tau \ln 2$. Work out $t_{1/2}$ for every draw, print its median and its central 68 % interval, and print the first-order value from the fitted width for comparison.

In [13]:
hbar = DescriptorNumber(name='hbar', value=0.6582119569, unit='meV*ps', variance=0.0)
two = DescriptorNumber(name='two', value=2.0, variance=0.0)
gamma_draws = columns['gamma']

<details class="question">
<summary>Hint 1</summary>

Compute the half-life for every draw at once with NumPy: `es.math` takes one number at a time, so it is not used on the draws. `np.percentile(..., [16, 50, 84])` gives the median and the central 68 % interval.

</details>

<details class="question">
<summary>Hint 2</summary>

The half-life is the relaxation time times $\ln 2$, and the relaxation time is $\hbar/\gamma$. For the first-order value use `es.math.log` on the constant `two` and the fitted width `fitted['gamma']`.

</details>

In [14]:
%%solution The two agree closely, because $\gamma$ is well determined and $t_{1/2}$ is nearly linear in it over its range.
t_half_draws = hbar.value / gamma_draws * np.log(2)
low, median, high = np.percentile(t_half_draws, [16, 50, 84])
show('half-life, first order', hbar / fitted['gamma'] * es.math.log(two))
print(f'half-life, posterior = {median:.4g} (+{high - median:.2g} / -{median - low:.2g}) ps')

## A poorly determined parameter

The resolution width $\sigma$ is known much less well than $\gamma$. A quantity that depends on its inverse,
such as the coherence time $\hbar/\sigma$ that belongs to the resolution width, is a strongly non-linear function of it. The
first-order uncertainty is symmetric, while the posterior is not:

In [15]:
coherence_first_order = hbar / fitted['sigma']
coherence_draws = hbar.value / columns['sigma']
low, median, high = np.percentile(coherence_draws, [16, 50, 84])

show('coherence time, first order', coherence_first_order)
print(f'coherence time, posterior = {median:.4g} (+{high - median:.2g} / -{median - low:.2g}) ps')

coherence time, first order = 605.3 ± 3.3e+02 ps
coherence time, posterior = 589.7 (+4.7e+02 / -2.1e+02) ps


In [16]:
limit = np.percentile(coherence_draws, 99)
grid = np.linspace(0.0, limit, 300)
gaussian = norm(coherence_first_order.value, coherence_first_order.error).pdf(grid)

fig = go.Figure()
fig.add_trace(
    go.Histogram(
        x=coherence_draws[coherence_draws < limit],
        histnorm='probability density',
        nbinsx=60,
        name='posterior',
        opacity=0.6,
    )
)
fig.add_trace(
    go.Scatter(x=grid, y=gaussian, mode='lines', name='first order', line=dict(color='black'))
)
fig.update_layout(
    xaxis_title='coherence time / ps',
    yaxis_title='probability density',
    bargap=0.02,
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.7, y=0.95),
)
show_figure(fig)

The posterior has a long tail towards long times: its central 68 % interval reaches much further above the
median than below it. The first-order interval is symmetric and cannot show this.

### Exercise 2: A quantity that is linear

The resolution full width at half maximum, $2\sqrt{2\ln 2}\;\sigma$, is proportional to $\sigma$. Print the standard deviation of its draws, in µeV, and the first-order uncertainty from `es.math`.

In [17]:
two = DescriptorNumber(name='two', value=2.0, variance=0.0)
sigma_draws = columns['sigma']

<details class="question">
<summary>Hint 1</summary>

On the draws use NumPy and the factor between $\sigma$ and the full width at half maximum. 1 meV is 1000 µeV.

</details>

<details class="question">
<summary>Hint 2</summary>

For the first-order value use `es.math.sqrt` and `es.math.log` on the constant `two` and on the fitted width `fitted['sigma']`, then `convert_unit('ueV')`.

</details>

In [18]:
%%solution For a linear function the two uncertainties are of the same size; the difference above came from the non-linearity.
fwhm_draws = 2 * np.sqrt(2 * np.log(2)) * sigma_draws * 1000
fwhm = 2 * es.math.sqrt(2 * es.math.log(two)) * fitted['sigma']
fwhm.convert_unit('ueV')
print(f'standard deviation of the draws = {fwhm_draws.std():.3g} µeV')
show('resolution FWHM, first order', fwhm)

### Exercise 3: How likely is a coarse resolution?

Count the fraction of the draws in which the resolution full width at half maximum, $2\sqrt{2\ln 2}\;\sigma$, is larger than 3 µeV. Print it as a percentage. This is the posterior probability of the statement.

In [19]:
sigma_draws = columns['sigma']

<details class="question">
<summary>Hint 1</summary>

Compute the draws of the full width at half maximum in µeV as in the previous exercise, and compare them with 3.

</details>

<details class="question">
<summary>Hint 2</summary>

The mean of a boolean array is the fraction of `True` values.

</details>

In [20]:
%%solution A posterior answers a question like this directly; a symmetric uncertainty can only suggest it.
fwhm_draws = 2 * np.sqrt(2 * np.log(2)) * sigma_draws * 1000
probability = (fwhm_draws > 3).mean()
print(f'probability that the resolution FWHM exceeds 3 µeV: {probability:.1%}')

## Good to know

* **The draws change from run to run.** DREAM is a random algorithm, so the numbers differ a little every
  time the notebook runs. Increasing `samples` makes them more stable, and the convergence check above says
  whether they can be trusted.
* **`es.math` is for fitted values; NumPy is for draws.** `es.math` works on one `Parameter` at a time and
  gives a first-order, symmetric uncertainty. For a quantity that is nearly linear in well-determined
  parameters it is a quick and accurate shortcut. For a strongly non-linear quantity of a poorly determined
  parameter, compute it on the draws.
* **The sampler moves the parameters.** After sampling, `model.gamma.value` is no longer the fitted value.
  That is why the fitted numbers were copied to `fitted` first.